# Predict account closure with client KumoTabular

Which accounts are likely to close voluntarily in the next 30 days? Use synthetic account aggregates and an existing KumoTabular NIM endpoint to evaluate later held-out accounts, score a prospective cohort, and save a reusable context.

Use Python 3.12 or newer and an endpoint supporting client sessions. See the [client prediction and session APIs](https://github.com/NVIDIA/structured-data-models-client/blob/main/docs/get-started/quickstart.md#one-table). The simulation illustrates a workflow; its metrics are **demonstration only**.

## Install and configure

Install the package into this notebook kernel.
Use an existing NIM endpoint serving the requested model; set `SDM_NIM_URL` and, when required, `SDM_API_KEY` in the environment.
No endpoint is provisioned by this notebook.

In [ ]:
%pip install structured-data-models-client pandas pyarrow scikit-learn

## Set the example parameters

Choose disjoint context, evaluation, and prospective cohorts. The outcome windows finish before the next cohort's cutoff; keep the threshold fixed before inspecting evaluation results.

In [ ]:
SEED = 1203
N_CONTEXT, N_TEST, N_SCORE = 2500, 1000, 500
CONTEXT_TIME = "2026-01-01"
TEST_TIME = "2026-02-15"
SCORE_TIME = "2026-04-01"
OBSERVATION_DAYS = 30
HORIZON_DAYS = 30
THRESHOLD = 0.5

SCRATCH_PARENT = None  # Use the operating system's temporary directory.
CLEANUP = False


Import the model/client and analysis libraries used by the following cells.


In [ ]:
import os
from pathlib import Path
from importlib.metadata import version
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

from sdm_client import KumoTabular, Request, SDMClient

Initialize the random generator. The printed directory is where this run will save predictions and context; existing directories are left alone.

In [ ]:
rng = np.random.default_rng(SEED)
owned_directory = tempfile.TemporaryDirectory(
    prefix="account-closure-", dir=SCRATCH_PARENT, delete=False
)
run_dir = Path(owned_directory.name)

print("Client:", version("structured-data-models-client"))
print("Outputs:", run_dir)

## Generate synthetic example data

Run the next cell to create account snapshots, historical aggregate features, and simulated future closure labels. The prospective cohort has unknown outcomes. You can skim the generation code; the model workflow starts at **Separate context, evaluation and scoring rows**.

In [ ]:
cohort_sizes = (N_CONTEXT, N_TEST, N_SCORE)
snapshot_times = pd.to_datetime(
    np.repeat([CONTEXT_TIME, TEST_TIME, SCORE_TIME], cohort_sizes)
)
cohort = np.repeat(["context", "test", "score"], cohort_sizes)
account_count = sum(cohort_sizes)
accounts = pd.DataFrame({
    "account_id": [f"synthetic_account_{i:05d}" for i in range(account_count)],
    "cohort": cohort,
    "snapshot_time": snapshot_times,
    "observation_start": snapshot_times - pd.Timedelta(days=OBSERVATION_DAYS),
    "last_observation_at": snapshot_times - pd.to_timedelta(rng.integers(1, 24, account_count), unit="h"),
})


accounts["product_type"] = rng.choice(["checking", "savings"], account_count)
accounts["segment"] = rng.choice(["standard", "plus", "student"], account_count)
accounts["region"] = rng.choice(["region_a", "region_b", "region_c"], account_count)
accounts["balance"] = rng.lognormal(7.4, 1.0, account_count).round(2)
accounts["inflow_30d"] = rng.lognormal(7.6, 0.7, account_count).round(2)
accounts["outflow_30d"] = rng.lognormal(7.4, 0.8, account_count).round(2)
accounts["transaction_count_30d"] = rng.poisson(20, account_count)
accounts["digital_login_count_30d"] = rng.poisson(5, account_count)
accounts["support_contacts_30d"] = rng.poisson(1.2, account_count)
accounts["monthly_fee"] = rng.choice([0.0, 5.0, 12.0, 20.0], account_count)
accounts["tenure_days"] = rng.integers(90, 3000, account_count)
display(accounts.head(3))


assert accounts.account_id.notna().all() and accounts.account_id.is_unique
assert (accounts.observation_start < accounts.last_observation_at).all()
assert (accounts.last_observation_at < accounts.snapshot_time).all()
assert accounts.tenure_days.gt(OBSERVATION_DAYS).all()
feature_columns = [
    "product_type", "segment", "region", "balance", "inflow_30d", "outflow_30d",
    "transaction_count_30d", "digital_login_count_30d", "support_contacts_30d",
    "monthly_fee", "tenure_days",
]
features = accounts[feature_columns].copy()
assert not features.isna().any().any()
assert np.isfinite(features.select_dtypes(include="number").to_numpy()).all()
print(f"Prepared {len(features)} accounts with {len(feature_columns)} features each.")


known_outcomes = accounts.cohort.ne("score")
horizon_end = accounts.snapshot_time + pd.Timedelta(days=HORIZON_DAYS)
accounts["observation_end"] = horizon_end.where(known_outcomes)
accounts["closed_at"] = pd.NaT
known_accounts = accounts.loc[known_outcomes]
log_odds = (-1.3 + 0.07 * known_accounts.monthly_fee
            - 0.25 * known_accounts.digital_login_count_30d
            + 0.55 * known_accounts.support_contacts_30d
            + rng.normal(0, 0.6, known_outcomes.sum()))
closes = rng.binomial(1, 1 / (1 + np.exp(-log_odds))).astype(bool)
closure_offsets = pd.to_timedelta(rng.integers(1, HORIZON_DAYS + 1, known_outcomes.sum()), unit="D")
closure_dates = (known_accounts.snapshot_time + closure_offsets).where(closes)
accounts.loc[known_outcomes, "closed_at"] = closure_dates
mature = accounts.observation_end.notna() & (accounts.observation_end >= horizon_end)
closed_in_horizon = accounts.closed_at.gt(accounts.snapshot_time) & accounts.closed_at.le(horizon_end)
labels = closed_in_horizon.astype(float).where(mature)
accounts["closure_in_horizon"] = labels
assert labels[cohort == "score"].isna().all()
assert set(labels[cohort == "context"]) == set(labels[cohort == "test"]) == {0, 1}
print(f"Known outcomes: {labels.notna().sum()}; prospective outcomes: {labels.isna().sum()}.")

accounts.to_csv(run_dir / "accounts.csv", index=False)


## Separate context, evaluation and scoring rows

Supply only the earlier context population's labels to the model. Reserve the later held-out accounts and their labels for evaluation; do not tune model settings or thresholds on them. Prospective accounts are scored without labels.

The context outcome window ends before the evaluation cutoff, and the evaluation outcome window ends before prospective scoring. The feature allowlist excludes all IDs, timestamps, population markers and future outcomes.


In [ ]:
context_mask = cohort == "context"
test_mask = cohort == "test"
score_mask = cohort == "score"
assert pd.Timestamp(CONTEXT_TIME) + pd.Timedelta(days=HORIZON_DAYS) < pd.Timestamp(TEST_TIME)
assert pd.Timestamp(TEST_TIME) + pd.Timedelta(days=HORIZON_DAYS) < pd.Timestamp(SCORE_TIME)

context_ids = set(accounts.loc[context_mask, "account_id"])
assert context_ids.isdisjoint(accounts.loc[test_mask | score_mask, "account_id"])
assert set(accounts.loc[test_mask, "account_id"]).isdisjoint(accounts.loc[score_mask, "account_id"])

context_df = features.loc[context_mask].reset_index(drop=True)
context_df["closure_in_horizon"] = labels[context_mask].astype(int).to_numpy()
test_df = features.loc[test_mask].reset_index(drop=True)
score_df = features.loc[score_mask].reset_index(drop=True)
y_test = labels[test_mask].astype(int).to_numpy()
heldout_ids = accounts.loc[test_mask, "account_id"].reset_index(drop=True)
requested_ids = accounts.loc[score_mask, "account_id"].reset_index(drop=True)
assert "closure_in_horizon" not in test_df and "closure_in_horizon" not in score_df
print(f"Context: {len(context_df)} | Held-out: {len(test_df)} | Prospective: {len(score_df)}")


Create two client requests with the same labeled context: one for evaluation, one for prospective scoring. Account IDs stay outside the feature frames and will be joined back to the responses.

In [ ]:
test_request = Request(
    context=context_df, predict=test_df,
    target="closure_in_horizon", task="classification",
)
score_request = Request(
    context=context_df, predict=score_df,
    target="closure_in_horizon", task="classification",
)

## Connect to the classifier

Use an existing NIM serving KumoTabular with session support.

In [ ]:
client = SDMClient(os.environ["SDM_NIM_URL"], api_key=os.environ.get("SDM_API_KEY"))

## Reuse labeled context

[Pin the earlier labeled accounts in a client session](https://github.com/NVIDIA/structured-data-models-client/blob/main/docs/get-started/quickstart.md#asking-the-same-context-several-questions).
Prospective and evaluation rows carry features only, and no pretrained weights are updated.

In [ ]:
fitted = client.fit(
    test_request, model=KumoTabular(), outputs=["prediction", "probabilities"]
)

Each response identifies its original request position in `row_index`. Join that value to the external account IDs instead of assuming response order. Check that every requested row appears once and that class probabilities are finite, bounded and sum to one.

In [ ]:
def checked_predictions(answer, account_ids):
    frame = answer.to_pandas()
    assert pd.api.types.is_integer_dtype(frame.row_index)
    assert frame.row_index.is_unique and set(frame.row_index) == set(range(len(account_ids)))
    probabilities = frame[["probability_0", "probability_1"]].to_numpy(dtype=float)
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
    identity = pd.DataFrame({"row_index": range(len(account_ids)), "account_id": account_ids})
    return identity.merge(frame, on="row_index", validate="one_to_one", sort=False)

test_result = checked_predictions(fitted.predict(test_request), heldout_ids)
p_test = test_result.probability_1.to_numpy()

Compare predictions with a constant baseline: the fraction of context accounts that closed. AUROC measures ranking; average precision depends on prevalence; Brier score measures probability error (lower is better). These held-out metrics are **demonstration only** and do not establish real-world performance.

In [ ]:
baseline = np.full(len(y_test), context_df.closure_in_horizon.mean())
metrics = pd.DataFrame({
    "KumoTabular": [roc_auc_score(y_test, p_test), average_precision_score(y_test, p_test),
                    brier_score_loss(y_test, p_test)],
    "Constant baseline": [roc_auc_score(y_test, baseline), average_precision_score(y_test, baseline),
                          brier_score_loss(y_test, baseline)],
}, index=["AUROC", "Average precision", "Brier score"])
assert np.isfinite(metrics.to_numpy()).all()
display(metrics.round(3))

## Batch predictions

Reuse the fitted context to score the prospective cohort. The result has one account ID and one closure probability per requested account. `above_threshold` is an illustrative review flag. Save the table for inspection; it is not an automated account decision.

In [ ]:
score_answer = fitted.predict(score_request)

### Inspect and save the prospective scores

Verify request positions against account IDs, inspect closure probabilities and the illustrative review flag, then save one row per requested account.

In [ ]:
score_result = checked_predictions(score_answer, requested_ids)
p_score = score_result.probability_1.to_numpy()
predictions = pd.DataFrame({
    "account_id": requested_ids,
    "closure_probability": p_score,
    "above_threshold": p_score >= THRESHOLD,
})
assert predictions.account_id.is_unique and predictions.account_id.tolist() == requested_ids.tolist()
predictions.to_parquet(run_dir / "predictions.parquet", index=False)
display(predictions.head(5))
print(f"Saved {len(predictions)} account predictions.")

## Save context and reproduce the scores

Save labeled context and prospective features with their account IDs. Reconnect and submit a new one-shot request containing that context; this does not recreate a persistent client session. Account IDs are bookkeeping and are removed before inference.


In [ ]:
context_df.to_parquet(run_dir / "context.parquet", index=False)
saved_scoring_rows = score_df.copy()
saved_scoring_rows.insert(0, "account_id", requested_ids)
saved_scoring_rows.to_parquet(run_dir / "scoring_rows.parquet", index=False)
print("Saved predictions, context and scoring rows in", run_dir)

Reload the files and resubmit their context in a new client connection. This same-process check compares IDs and probabilities; it is not a serialized-model or fresh-process test.

In [ ]:
restored_context = pd.read_parquet(run_dir / "context.parquet")
restored_scoring_rows = pd.read_parquet(run_dir / "scoring_rows.parquet")
restored_ids = restored_scoring_rows.pop("account_id")
restored_request = Request(
    context=restored_context, predict=restored_scoring_rows,
    target="closure_in_horizon", task="classification",
)
with SDMClient(os.environ["SDM_NIM_URL"], api_key=os.environ.get("SDM_API_KEY")) as restored_client:
    reloaded_predictions = checked_predictions(
        restored_client.predict(restored_request, model=KumoTabular(), outputs=["probabilities"]),
        restored_ids,
    )
assert reloaded_predictions.account_id.tolist() == requested_ids.tolist()
reconstruction_matches = np.allclose(reloaded_predictions.probability_1, p_score, rtol=1e-5, atol=1e-6)
assert reconstruction_matches
print("Rebuilt client context reproduces all prospective predictions.")

## Cleanup

Close the fitted session and client. Set `CLEANUP=True` to remove only this run's temporary output directory; the default keeps its files for inspection.

In [ ]:
fitted.close()
client.close()
if CLEANUP:
    owned_directory.cleanup()
    print("Removed this run's output directory.")
else:
    print("Saved files remain in", run_dir)